# Model Based Collaborative Filtering using Matrix Factorization (SVD)

In this notebook, We implement model-based collaborative filtering
using Singular Value Decomposition (SVD).

Unlike memory-based approaches, this method decomposes the
user-item interaction matrix into latent factors that capture
hidden user preferences and item characteristics.

This approach handles sparsity efficiently and scales better
for large datasets.

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error

In [3]:
from google.colab import drive
drive.mount('/content/drive')

base_path = "/content/drive/MyDrive/Goodreads-Book-Recommendation-System/data/"

books = pd.read_csv(base_path + "books.csv")
ratings = pd.read_csv(base_path + "ratings.csv")

print("Books shape:", books.shape)
print("Ratings shape:", ratings.shape)

Mounted at /content/drive
Books shape: (10000, 23)
Ratings shape: (981756, 3)


# User-Book Matrix

In [5]:
# Using subset of users for easy computation
sample_users = ratings['user_id'].unique()[:5000]

ratings_subset = ratings[ratings['user_id'].isin(sample_users)]

interaction_matrix = ratings_subset.pivot_table(
    index='user_id',
    columns='book_id',
    values='rating'
).fillna(0)

interaction_matrix.shape

(5000, 9907)

# Applying SVD

In [7]:
from sklearn.decomposition import TruncatedSVD

# Convert to numpy
R = interaction_matrix.values

# Truncated SVD
k = 50

svd = TruncatedSVD(n_components=k, random_state=42)
R_reduced = svd.fit_transform(R)

# Reconstruct approximated matrix
R_pred = np.dot(R_reduced, svd.components_)

Instead of computing full SVD, truncated SVD was used to reduce computational complexity and make the model scalable to larger datasets.

# Evaluate Model
We evaluate only on non-zero ratings.

In [9]:
# Get actual and predicted values for rated items only

actual = R[R > 0]
predicted = R_pred[R > 0]

rmse = np.sqrt(mean_squared_error(actual, predicted))

print("RMSE:", rmse)

RMSE: 3.3377287605086625


# Recommendation Function

In [14]:
def recommend_books_svd(user_id, n=5):

    if user_id not in interaction_matrix.index:
        return "User not in subset."

    # Get user index position
    user_index = interaction_matrix.index.get_loc(user_id)

    # Get user's actual ratings
    user_ratings = interaction_matrix.loc[user_id]

    # Get predicted ratings
    predicted_ratings = R_pred[user_index]

    # Create dataframe
    recommendations = pd.DataFrame({
        'book_id': interaction_matrix.columns,
        'predicted_rating': predicted_ratings
    })

    # Remove already rated books properly
    rated_books = user_ratings[user_ratings > 0].index

    recommendations = recommendations[
        ~recommendations['book_id'].isin(rated_books)
    ]

    # Sort
    recommendations = recommendations.sort_values(
        by='predicted_rating',
        ascending=False
    )

    top_books = recommendations.head(n)

    return books[
        books['book_id'].isin(top_books['book_id'])
    ]['title']

In [15]:
recommend_books_svd(sample_users[0])

,title
1975,I'm a Stranger Here Myself: Notes on Returning...
2278,Neither Here nor There: Travels in Europe
6843,Moon Palace
